# Lesson 10 · Radioactive decay

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/10_radioactive_decay.ipynb)

**For:** A-level Physics (for example AQA 3.8.1.3, OCR A 6.4.4, Edexcel Topic 11) and IB Physics (E.3 radioactive decay). Check your own specification for the exact wording.

**Before this lesson:** Lesson 1, *Simple harmonic motion* (exponential decay, first met in damped oscillations). Also assumed: the structure of the nucleus and alpha, beta and gamma radiation.

**Leads to:** Lesson 11, *Capacitors*; Lesson 17, *Nuclear binding energy*.

**You will be able to:**

1. explain why radioactive decay is random for one nucleus but predictable for many
2. use $N = N_0e^{-\lambda t}$, $A = \lambda N$ and $t_{1/2} = \ln 2/\lambda$
3. find a half-life from data using a graph of $\ln N$ against $t$
4. describe the statistical spread in counts, and why bigger samples give more reliable results
5. explain radioactive dating and calculate an age; describe a simple decay chain

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

[Try it in the explorer](../explorer/#decay).

In [ ]:
# Install the course API in a fresh Colab runtime or upgrade an older installation.
import importlib.util
import subprocess
import sys
from importlib.metadata import version

if importlib.util.find_spec("qplay") is None or int(version("quantum-playground").split(".")[0]) < 1:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quantum-playground>=1,<2"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import nuclear

## 1. Random, but predictable

An unstable nucleus can decay at any moment. Nothing about its history or surroundings changes that: each nucleus simply has a fixed probability $\lambda\,\Delta t$ of decaying in the next short time $\Delta t$. $\lambda$ is the **decay constant**. You cannot say *which* nucleus will decay next, or when.

Yet a large sample behaves very predictably. The simulation below follows every nucleus individually, deciding at random in each time step whether it decays.

**Predict (P1).** You start with 20 nuclei, and separately with 20 000 nuclei, each with a half-life of 1 minute. After exactly 1 minute, will exactly half of each sample be left? Write your prediction down before you run the next cell.

In [ ]:
half_life = 1.0                                  # minutes
lam = nuclear.decay_constant(half_life)
t = np.linspace(0, 4, 81)
small = nuclear.simulate_decay(20, lam, t, trials=6, seed=1)          # six separate runs of 20 nuclei
large = nuclear.simulate_decay(20_000, lam, t, trials=1, seed=2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for run in small:
    ax1.step(t, run / 20, where="post", alpha=0.7)
ax1.plot(t, np.exp(-lam * t), "k--", label=r"$e^{-\lambda t}$")
ax1.set(xlabel="t / min", ylabel="fraction left", title="20 nuclei, six runs"); ax1.legend()
ax2.plot(t, large[0] / 20_000, label="20 000 nuclei")
ax2.plot(t, np.exp(-lam * t), "k--", label=r"$e^{-\lambda t}$")
ax2.set(xlabel="t / min", title="20 000 nuclei, one run"); ax2.legend()
one = np.searchsorted(t, 1.0)
print("left after 1 minute, 20 nuclei, six runs:", small[:, one])
print(f"left after 1 minute, 20 000 nuclei: {large[0, one]} ({large[0, one] / 20_000:.3%})")

**Explain (P1).** Why does the large sample follow the smooth curve so much more closely than the small one?

For a large number of nuclei $N$, the number decaying per second is proportional to how many are left:

$$\frac{dN}{dt} = -\lambda N\quad\Rightarrow\quad N = N_0e^{-\lambda t},\qquad A = \lambda N,\qquad t_{1/2} = \frac{\ln 2}{\lambda}.$$

The **activity** $A$ (decays per second) is measured in becquerels, Bq. The **half-life** $t_{1/2}$ is the time for $N$, or $A$, to halve.

## 2. Half-life and the log graph

Every half-life, whatever is left halves again.

**Predict (P2).** After three half-lives, what fraction of the original nuclei is left: $\tfrac13$, $\tfrac16$ or $\tfrac18$? Write your prediction down before you run the next cell.

In [ ]:
half_life = 6.0                                  # hours, about the half-life of technetium-99m   <- Try it
lam = nuclear.decay_constant(half_life)
t = np.linspace(0, 30, 31)
counts = nuclear.simulate_decay(5000, lam, t, seed=3)[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(t, counts, "o", ms=4, label="simulated")
ax1.plot(t, nuclear.remaining(5000, lam, t), "k--", label=r"$N_0e^{-\lambda t}$")
for k in range(1, 5):
    ax1.axvline(k * half_life, color="0.7", lw=0.8)
    ax1.text(k * half_life, 5000 * 0.5**k + 150, f"1/{2**k}", ha="center", fontsize=8)
ax1.set(xlabel="t / h", ylabel="N", title="Halving every 6 hours"); ax1.legend()

slope, intercept = np.polyfit(t, np.log(counts), 1)       # ln N = ln N0 - lambda t
ax2.plot(t, np.log(counts), "o", ms=4)
ax2.plot(t, intercept + slope * t, "k--", label=f"best fit: gradient {slope:.4f} per hour")
ax2.set(xlabel="t / h", ylabel="ln N", title="A straight line on a log graph"); ax2.legend()
print(f"after 3 half-lives: {counts[18]} of 5000 left = {counts[18] / 5000:.3f}")
print(f"half-life from the fit: ln2/{-slope:.4f} = {np.log(2) / -slope:.2f} h (true value {half_life} h)")

**Explain (P2).** Why is $\ln N$ against $t$ a straight line, and how does its gradient give the half-life?

**Exam tip:** to find a half-life from data, plot $\ln N$ (or $\ln A$, or the log of a count rate) against $t$. The gradient is $-\lambda$, and $t_{1/2} = \ln 2/\lambda$. This uses every point, so it is far more reliable than reading one halving off a curved graph.

## 3. How random is "random"?

Repeat the same experiment many times and the number left after a fixed time varies. For $N_0$ nuclei each surviving with probability $p = e^{-\lambda t}$, the spread follows the **binomial** distribution, with standard deviation $\sqrt{N_0\,p\,(1 - p)}$. This is why count rates in the lab fluctuate, and why longer counts give more reliable results.

In [ ]:
N0, lam = 500, 0.2                               # <- Try it: bigger N0
t = np.array([0.0, np.log(2) / lam])             # look at one half-life, p = 1/2
left = nuclear.simulate_decay(N0, lam, t, trials=2000, seed=4)[:, 1]
p = np.exp(-lam * t[1])
plt.figure(figsize=(7, 3.6))
plt.hist(left, bins=np.arange(left.min() - 0.5, left.max() + 1.5), density=True, alpha=0.6, label="2000 repeats")
n = np.arange(left.min(), left.max() + 1)
sd = np.sqrt(N0 * p * (1 - p))
plt.plot(n, np.exp(-(n - N0 * p) ** 2 / (2 * sd**2)) / (sd * np.sqrt(2 * np.pi)), "k", label="normal curve with binomial spread")
plt.xlabel("nuclei left after one half-life"); plt.legend()
print(f"mean {left.mean():.1f} (expected {N0 * p:.1f});  standard deviation {left.std():.2f} (expected {sd:.2f})")

## 4. Decay chains

Often the **daughter** nucleus is radioactive too. Radon-222, for example, comes from a long chain starting at uranium-238. For a parent decaying into a radioactive daughter and then a stable nucleus,

$$\frac{dN_1}{dt} = -\lambda_1N_1,\qquad \frac{dN_2}{dt} = \lambda_1N_1 - \lambda_2N_2 .$$

The daughter is made by the parent's decays and lost by its own. `nuclear.bateman` solves these equations exactly (the Bateman solution). This goes beyond most specifications, but it explains how short-lived nuclei like radon can be found in nature at all.

**Predict (P3).** A long-lived parent (half-life 1000 h) decays into a short-lived daughter (half-life 2 h). Starting with pure parent, how will the daughter's *activity* compare with the parent's after a day: much smaller, about equal, or much larger? Write your prediction down before you run the next cell.

In [ ]:
lam1, lam2 = nuclear.decay_constant(1000.0), nuclear.decay_constant(2.0)
t = np.linspace(0, 30, 301)
N = nuclear.bateman(1e6, [lam1, lam2, 0.0], t)          # parent, daughter, stable end product
A1, A2 = lam1 * N[:, 0], lam2 * N[:, 1]

plt.figure(figsize=(7, 3.8))
plt.plot(t, A1, label="parent activity")
plt.plot(t, A2, label="daughter activity")
plt.xlabel("t / h"); plt.ylabel("activity / decays per hour"); plt.legend()
plt.title("Daughter activity catches up with the parent's")
print(f"after 24 h: daughter activity / parent activity = {A2[240] / A1[240]:.4f}")
print(f"Bateman formula agrees with an independent matrix-exponential solution to "
      f"{np.max(np.abs(N - nuclear.decay_chain(1e6, [lam1, lam2, 0.0], t))):.1e} nuclei")

**Explain (P3).** Why do the two activities end up equal? (Think about what happens to the daughter when it is made faster than it decays.)

## 5. Radioactive dating

Living things take in carbon-14 from the air, so their carbon has a fixed fraction of it. After death no more is taken in, and the carbon-14 decays with a half-life of 5730 years. Measuring how much activity is left gives the age:

$$\frac{A}{A_0} = e^{-\lambda t}\quad\Rightarrow\quad t = \frac{1}{\lambda}\ln\frac{A_0}{A}.$$

In [ ]:
lam_c14 = nuclear.decay_constant(5730.0)          # per year
ages = np.linspace(0, 40_000, 400)
plt.figure(figsize=(7, 3.6))
plt.plot(ages, nuclear.remaining(100, lam_c14, ages))
measured = 25.0                                   # per cent of the original activity   <- Try it
age = np.log(100 / measured) / lam_c14
plt.plot(age, measured, "o", color="C3")
plt.annotate(f"{measured:.0f}% left: {age:,.0f} years", (age, measured), xytext=(10, 10), textcoords="offset points")
plt.xlabel("age / years"); plt.ylabel("carbon-14 activity left / %"); plt.title("Carbon dating");

Carbon dating works for ages up to about 50 000 years; after roughly ten half-lives too little is left to measure. Rocks are dated the same way with much longer-lived nuclei such as uranium-238 (half-life 4.5 billion years).

---

## Exercises

Give answers to 3 significant figures.

**Q1.** Iodine-131 has a half-life of 8.0 days. Find its decay constant in s$^{-1}$.

**Q2.** A sample contains $2.0\times10^{15}$ iodine-131 nuclei. Find its activity.

**Q3.** What fraction of the iodine-131 is left after 20 days?

**Q4.** A wooden bowl has 30% of the carbon-14 activity of living wood. Carbon-14 has a half-life of 5730 years. How old is the bowl?

**Q5.** Cobalt-60 has a half-life of 5.27 years. How long does it take for a source's activity to fall to 1.0% of its starting value?

**Q6 (simulation).** 1000 nuclei have decay constant $\lambda = 0.10\ \mathrm{s^{-1}}$. Use `nuclear.simulate_decay(1000, 0.10, t, trials=2000)` to estimate the standard deviation of the number left after 10 s. (Any answer within 10% of the exact value counts.)

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_lam1 = nuclear.decay_constant(8.0 * 86400)
_p6 = np.exp(-0.10 * 10)
_answers = {
    "Q1": _lam1,
    "Q2": _lam1 * 2.0e15,
    "Q3": nuclear.remaining(1.0, nuclear.decay_constant(8.0), 20.0),
    "Q4": np.log(1 / 0.30) / nuclear.decay_constant(5730.0),
    "Q5": np.log(100) / nuclear.decay_constant(5.27),
    "Q6": np.sqrt(1000 * _p6 * (1 - _p6)),                # binomial standard deviation
}
_tol = {"Q6": 0.10}                                       # a simulated estimate is itself random

# Replace None with your answers, in SI units (Q3 as a decimal fraction, Q4 and Q5 in years).
mine = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q], _tol.get(q, 0.01))

---

## Worked solutions

Try the questions before reading on.

**Q1.** $t_{1/2} = 8.0 \times 86400 = 6.91\times10^{5}$ s, so $\lambda = \ln 2/t_{1/2} = 1.00\times10^{-6}\ \mathrm{s^{-1}}$.

**Q2.** $A = \lambda N = 1.003\times10^{-6} \times 2.0\times10^{15} = 2.01\times10^{9}$ Bq.

**Q3.** 20 days is $20/8 = 2.5$ half-lives, so the fraction left is $(\tfrac12)^{2.5} = 0.177$.

**Q4.** $t = \dfrac{1}{\lambda}\ln\dfrac{A_0}{A} = \dfrac{5730}{\ln 2}\ln\dfrac{1}{0.30} = 9.95\times10^{3}$ years, about 10 000 years.

**Q5.** $t = \dfrac{\ln 100}{\lambda} = 5.27 \times \dfrac{\ln 100}{\ln 2} = 35.0$ years. (That is $\log_2 100 = 6.64$ half-lives.)

**Q6.** Each nucleus survives 10 s with probability $p = e^{-1} = 0.368$, so the exact standard deviation is $\sqrt{1000 \times 0.368 \times 0.632} = 15.2$. The cell below runs the simulation.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}")
_left = nuclear.simulate_decay(1000, 0.10, np.array([0.0, 10.0]), trials=2000, seed=5)[:, 1]
print(f"simulated standard deviation: {_left.std():.2f}")

**Next:** in section 3, change `N0` to 50 and then 50 000. The spread grows, but the spread *as a fraction of the mean* shrinks. By how much does it shrink when $N_0$ goes up 100 times?

### Predict and explain: answers

**P1.** No: about half, but not exactly. With 20 nuclei the runs scatter widely, while 20 000 nuclei land within about 1% of half. The spread in the number left is about $\sqrt{N}$, so the *fractional* spread is about $1/\sqrt N$: around 22% for 20 nuclei but only 0.7% for 20 000. Randomness at the level of single nuclei becomes a precise law for large numbers.

**P2.** $\tfrac18$: each half-life halves what is left, so three give $(\tfrac12)^3$. Taking logs of $N = N_0e^{-\lambda t}$ gives $\ln N = \ln N_0 - \lambda t$, a straight line with gradient $-\lambda$; then $t_{1/2} = \ln 2/\lambda$. The fit recovers the 6 h half-life from noisy simulated data.

**P3.** About equal: the printed ratio is 1.002 after a day. At first the daughter is made faster than it decays, so it builds up. But its decay rate $\lambda_2N_2$ grows as it builds up, until it decays exactly as fast as it is made: $\lambda_2N_2 = \lambda_1N_1$. This is **secular equilibrium**. It takes a few daughter half-lives to reach, and then lasts as long as the parent does. (The ratio is a shade above 1, at $\lambda_2/(\lambda_2 - \lambda_1) = 1.002$, because the parent is itself decaying slowly.)